## Text Classification - MLP

## setup

In [ ]:
import torch 
import torch.nn as nn
from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.trainers import WordLevelTrainer
from tokenizers.pre_tokenizers import Whitespace

corpus = [
    "gậy ông đập lưng ông",
    "có làm mới có ăn"
]
data_size = len(corpus)

# 0: negative - 1: positive
labels = [0, 1]

# Define the max vocabulary size and sequence length
vocab_size = 8
sequence_length = 5

In [2]:
# Initialize the tokenizer and define a trainer
tokenizer = Tokenizer(WordLevel())
tokenizer.pre_tokenizer = Whitespace()
tokenizer.enable_padding(pad_id=1, 
                         pad_token="<pad>", 
                         length=sequence_length)
tokenizer.enable_truncation(max_length=sequence_length)

# Train the tokenizer on your corpus
trainer = WordLevelTrainer(vocab_size=vocab_size, 
                           special_tokens=["<unk>", "<pad>"])
tokenizer.train_from_iterator(corpus, trainer)


# Vectorize the samples
corpus_ids = []
for sentence in corpus:
    output = tokenizer.encode(sentence)
    output = torch.tensor(output.ids, dtype=torch.long)
    corpus_ids.append(output)

In [3]:
for v in corpus_ids:
    print(v)

tensor([4, 3, 0, 6, 3])
tensor([2, 5, 7, 2, 0])


In [ ]:
embedding = nn.Embedding(vocab_size, 2)  # (5, 2)
flatten = nn.Flatten()
classifier = nn.Linear(5*2, 2)           # (2,)

model = nn.Sequential(embedding, flatten, classifier)

In [6]:
from torchinfo import  summary

print(model)

Sequential(
  (0): Embedding(8, 2)
  (1): Flatten(start_dim=1, end_dim=-1)
  (2): Linear(in_features=10, out_features=2, bias=True)
)


In [7]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(),
                             lr=0.1)

## Train

In [8]:
inputs = torch.tensor([[4, 3, 0, 6, 3],
                       [2, 5, 7, 2, 0]], dtype=torch.long)
labels = torch.tensor([0, 1], dtype=torch.long)

for _ in range(50):
    optimizer.zero_grad()
    outputs = model(inputs)
    loss = criterion(outputs, labels)
    print(loss.item())

    loss.backward()
    optimizer.step()

1.139449119567871
0.5304548144340515
0.282034307718277
0.18193009495735168
0.13275201618671417
0.10416024923324585
0.08558358997106552
0.0725736916065216
0.06296361982822418
0.055578239262104034
0.04972687363624573
0.044977445155382156
0.04104605317115784
0.03773847222328186
0.0349174439907074
0.032483212649822235
0.030361413955688477
0.028495773673057556
0.02684260904788971
0.025367703288793564
0.024043813347816467
0.0228489488363266
0.021765103563666344
0.020777627825737
0.01987420581281185
0.0190446600317955
0.0182802751660347
0.017573678866028786
0.016918640583753586
0.01630965620279312
0.015742138028144836
0.01521200593560934
0.014715633355081081
0.014249966479837894
0.013812238350510597
0.013400021940469742
0.01301118079572916
0.01264374703168869
0.01229603961110115
0.011966554448008537
0.011653898283839226
0.011356677860021591
0.011074023321270943
0.010804710909724236
0.010547870770096779
0.010302746668457985
0.010068463161587715
0.009844263084232807
0.009629681706428528
0.00942

## Check if the model has adready learned sucessfuly

In [9]:
outputs = model(inputs)
print(outputs)
print(torch.softmax(outputs, axis=-1))

tensor([[ 1.9077, -2.5480],
        [-2.9289,  2.0428]], grad_fn=<AddmmBackward0>)
tensor([[0.9885, 0.0115],
        [0.0069, 0.9931]], grad_fn=<SoftmaxBackward0>)
